In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dtale
from pandas.io.xml import preprocess_data
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder,Binarizer, TargetEncoder

In [2]:
train_data = pd.read_csv('data/train.csv')
test_data = pd.read_csv('data/test.csv')

In [3]:
# d = dtale.show(train_data)
# d.open_browser()

Проводим исследование, осматриваем данные, убираем лишние столбцы

In [4]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

In [5]:
numeric_features = ['LotFrontage','LotArea','MasVnrArea','TotalBsmtSF','GrLivArea','GarageYrBlt','GarageArea','WoodDeckSF','OpenPorchSF','EnclosedPorch','ScreenPorch','PoolArea','MiscVal','YearBuilt','1stFlrSF','2ndFlrSF','Fireplaces','BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd']
nominal_features = ['MSSubClass','MSZoning','LotShape','LotConfig','Condition1','BldgType','HouseStyle','RoofStyle','RoofMatl','MasVnrType','Heating','Electrical','Neighborhood','Functional','GarageType','GarageFinish','PavedDrive','SaleType','SaleCondition','Exterior1st','Street','Alley','CentralAir']
ordinal_features = ['LandContour','Utilities','LandSlope','OverallQual','OverallCond','BsmtQual','BsmtExposure','HeatingQC','KitchenQual','GarageQual','ExterQual','ExterCond']

In [6]:
# Удаляем дубликаты и не нужные столбцы
train_data.drop_duplicates(inplace=True)
test_data.drop_duplicates(inplace=True)

test_ids = test_data['Id']
train_data = train_data.drop(['Id','Fence','MiscFeature'], axis=1)
test_data= test_data.drop(['Id','Fence','MiscFeature'], axis=1)

In [7]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 78 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   str    
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   str    
 5   Alley          91 non-null     str    
 6   LotShape       1460 non-null   str    
 7   LandContour    1460 non-null   str    
 8   Utilities      1460 non-null   str    
 9   LotConfig      1460 non-null   str    
 10  LandSlope      1460 non-null   str    
 11  Neighborhood   1460 non-null   str    
 12  Condition1     1460 non-null   str    
 13  Condition2     1460 non-null   str    
 14  BldgType       1460 non-null   str    
 15  HouseStyle     1460 non-null   str    
 16  OverallQual    1460 non-null   int64  
 17  OverallCond    1460 non-null   int64  
 18  YearBuilt      1460

In [8]:
test_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 77 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1459 non-null   int64  
 1   MSZoning       1455 non-null   str    
 2   LotFrontage    1232 non-null   float64
 3   LotArea        1459 non-null   int64  
 4   Street         1459 non-null   str    
 5   Alley          107 non-null    str    
 6   LotShape       1459 non-null   str    
 7   LandContour    1459 non-null   str    
 8   Utilities      1457 non-null   str    
 9   LotConfig      1459 non-null   str    
 10  LandSlope      1459 non-null   str    
 11  Neighborhood   1459 non-null   str    
 12  Condition1     1459 non-null   str    
 13  Condition2     1459 non-null   str    
 14  BldgType       1459 non-null   str    
 15  HouseStyle     1459 non-null   str    
 16  OverallQual    1459 non-null   int64  
 17  OverallCond    1459 non-null   int64  
 18  YearBuilt      1459

In [9]:
X = train_data.drop('SalePrice',axis=1)
y = np.log1p(train_data['SalePrice'])

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        # Numeric медиана + scaling
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]),numeric_features),

        # Nominal - OHE + обработка пропусков
        ('nominal', OneHotEncoder(handle_unknown='ignore', sparse_output=False),nominal_features),

        # Ordinal : мода + OE
        ('ordinal', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)),
        ]),ordinal_features),

        # Кодирование очень большого признака
        ('target_enc', TargetEncoder(), ['Neighborhood']),
    ],
    remainder='drop'
)

In [11]:
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureCreator(BaseEstimator,TransformerMixin):
    def __init__(self):
        self.bad_condition = ['Artery','Feedr','RRNn','RRAn','RRNe','RRAe']
        self.quality_materials = {
            'Stone': 3, 'BrkFace': 2, 'CemntBd': 1, 'VinylSd': 1
        }

    def fit(self,X, y=None):
        return self

    def transform(self,X):
        X_new = X.copy()
        # Новые признаки:

        # Выявление близкости к дороге и шуму
        X_new['HasBadCondition'] = (
            X_new['Condition1'].fillna('None').isin(self.bad_condition) |
            X_new['Condition2'].fillna('None').isin(self.bad_condition)
        ).astype(int)
        X_new = X_new.drop('Condition2', axis=1)

        # Вычисление возраста дома
        X_new['HouseAge'] = X_new['YrSold'] - X_new['YearBuilt']
        # Наличие ремонта в доме
        X_new['RemodAge'] = X_new['YrSold'] - X_new['YearRemodAdd']

        # Вычисление дорого материала для фасада крыши
        ext1 = X_new['Exterior1st'].fillna('None').map(self.quality_materials).fillna(0)
        ext2 = X_new['Exterior2nd'].fillna('None').map(self.quality_materials).fillna(0)
        X_new['GoodExterior'] = np.maximum(ext1, ext2)

        # Вычисление всей площади дома + подвал
        X_new['TotalArea'] = (X_new['TotalBsmtSF'].fillna(0) + X_new['1stFlrSF'].fillna(0) + X_new['2ndFlrSF'].fillna(0))

        # Количетсво санузлов в доме
        X_new['TotalBaths'] = (
            X_new['BsmtFullBath'].fillna(0) +
            X_new['FullBath'].fillna(0) +
            (
                    0.5 * X_new['BsmtHalfBath'].fillna(0) + X_new['HalfBath'].fillna(0)
             )
        )

        return X_new

In [12]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

pipeline = Pipeline([
    ('features', FeatureCreator()),
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0))
])

# Тестирование
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
rmse = np.sqrt(-scores.mean())
print(f"RMSE CV: {rmse:.4f}")

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE на тесте: {rmse_test:.4f}")


RMSE CV: 0.1525
RMSE на тесте: 0.1323


In [13]:
from sklearn.ensemble import RandomForestRegressor

pipeline02 = Pipeline([
    ('features', FeatureCreator()),
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

pipeline02.fit(X_train, y_train)
y_pred_02 = pipeline02.predict(X_test)

scores = cross_val_score(pipeline02, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
rmse = np.sqrt(-scores.mean())
print(f"RMSE CV: {rmse:.4f}")

rmse_test02 = np.sqrt(mean_squared_error(y_test, y_pred_02))
print(f"RMSE на тесте: {rmse_test02:.4f}")


RMSE CV: 0.1468
RMSE на тесте: 0.1469


In [14]:
from sklearn.ensemble import GradientBoostingRegressor

pipeline03 = Pipeline([
    ('features', FeatureCreator()),
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=42))
])

pipeline03.fit(X_train, y_train)
y_pred_03 = pipeline03.predict(X_test)

scores = cross_val_score(pipeline03, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
rmse = np.sqrt(-scores.mean())
print(f"RMSE CV: {rmse:.4f}")

rmse_test03 = np.sqrt(mean_squared_error(y_test, y_pred_03))
print(f"RMSE на тесте: {rmse_test03:.4f}")


RMSE CV: 0.1332
RMSE на тесте: 0.1372


In [15]:
pipeline03.fit(X, y)
y_test_pred = pipeline03.predict(test_data)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': np.expm1(y_test_pred)
})

submission.to_csv('submission_new_features_03.csv', index=False)

In [18]:
# Построение нейронной сети
import torch
import typing
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR

# Задаем random seed
np.random.seed(42)
torch.manual_seed(42)

def define_parameters(batch_size=32,epochs=300,lr=1e-3):
    return {'batch_size':batch_size, 'epochs': epochs, 'lr': lr}

def define_hyperparameters(criterion,optimizer,scheduler=None,epochs=100,patience=10,device='cpu',verbose=True):
    return {
        'criterion': criterion,
        'optimizer': optimizer,
        'scheduler': scheduler,
        'patience': patience,
        'epochs': epochs,
        'device': device,
        'verbose': verbose
    }

class HousePricePredictor(nn.Module):
    def __init__(self, input_dim, output_dim=1, hidden_dim=[64,32], use_dropout=False, use_batch_norm=True, activation='relu'):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dim:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())

            if use_dropout:
                layers.append(nn.Dropout(p=0.5))

            if use_batch_norm:
                layers.append(nn.BatchNorm1d(h))

            if activation == 'relu':
                layers.append(nn.ReLU())
            elif activation == 'tanh':
                layers.append(nn.Tanh())
            elif activation == 'leaky_relu':
                layers.append(nn.LeakyReLU(0.1))
            else:
                raise ValueError("Unsupported activation")
            prev_dim = h
        layers.append(nn.Linear(prev_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self,x):
        return self.net(x)

64


In [17]:
def train_model(model,train_loader,test_loader,criterion, optimizer,scheduler=None,epochs=100,patience=10,device='cpu',verbose=True):
  model.to(device)
  train_losses, test_losses = [],[]
  train_accs, test_accs = [],[]

  best_test_acc = 0
  best_model_state = None
  patience_counter = 0
  for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
      inputs, labels = inputs.to(device), labels.to(device)
      optimizer.zero_grad()
      outputs = model(inputs)
      loss = criterion(outputs,labels)
      loss.backward()
      optimizer.step()
      train_loss += loss.item() * inputs.size(0)
      _, predicted = torch.max(outputs, 1)
      total_train += labels.size(0)
      correct_train += (predicted == labels).sum().item()

    avg_train_loss = train_loss / total_train
    train_acc = 100 * correct_train / total_train
    train_losses.append(avg_train_loss)
    train_accs.append(train_acc)

    model.eval()
    test_loss = 0.0
    correct_test = 0
    total_test = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
      for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs,labels)
        test_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

      avg_test_loss = test_loss / total_test
      test_acc = 100 * correct_test / total_test
      test_losses.append(avg_test_loss)
      test_accs.append(test_acc)

      if scheduler:
        if isinstance(scheduler, ReduceLROnPlateau):
            scheduler.step(avg_test_loss)
        else:
            scheduler.step()

      if test_acc > best_test_acc:
          best_test_acc = test_acc
          best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
          patience_counter = 0
      else:
          patience_counter += 1
          if patience_counter >= patience:
              if verbose:
                  print(f"Early stopping at epoch {epoch+1}")
                  break

      if verbose and (epoch+1) % 20 == 0:
          print(f"Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

  model.load_state_dict(best_model_state)
  return model, train_losses, test_losses, train_accs, test_accs, all_preds, all_labels
